# Module 1 — Planting Window (maize, GHA)
Estimates the **planting dekad** per maize pixel: cue-fusion green-up (Sentinel-2 NDRE + S1 SAR + FPAR) for the main seasons, or CHIRPS rainfall onset (25/20 mm) for the short rains, then the inception-report **5+7 false-start gate**.

**Where to start:** put the `planting_pipeline` folder on your Google Drive, run the cells top-to-bottom, and approve the Drive-mount and Earth-Engine sign-in prompts.

## Setup

In [ ]:
!pip -q install earthengine-api geemap pandas geopandas 2>/dev/null
print('installed.')

In [ ]:
import ee
PROJECT="ee-manzikye"
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT)
print("EE ready:", ee.String("ok").getInfo())

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import sys, os
PIPE_DIR="/content/drive/MyDrive/planting_pipeline"   # adjust if needed
assert os.path.isdir(PIPE_DIR), f"Upload planting_pipeline to Drive; not at {PIPE_DIR}"
sys.path.insert(0, PIPE_DIR); os.chdir(PIPE_DIR)
print("pipeline on path:", PIPE_DIR)

In [ ]:
# --- config + GEE-native map (geemap: built-in EE Layers panel, toggle + opacity) ---
COUNTRY="Kenya"      # "Kenya" | "Ethiopia"
SEASON ="Long rains" # "Long rains" | "Short rains" | "Meher"
YEAR=2024
S1_ORBIT="ASCENDING"   # S1B gone (2022) -> ASCENDING has coverage over Kenya
from run import GAUL_NAME
from src import zonal_aggregate as ZA
aoi = ZA.gaul_admin(ee, [GAUL_NAME[COUNTRY]], level=0).geometry()
aoi_run = ee.Geometry.Rectangle([34.4,-1.2,37.8,1.2])   # fast test box; use `aoi` for whole country
import geemap
try:
    from google.colab import output; output.enable_custom_widget_manager()  # needed for interactive geemap in Colab
except Exception:
    pass
def new_map(zoom=7):
    m = geemap.Map(add_google_map=False, basemap="SATELLITE")  # keyless Google tiles; native EE layer control
    m.centerObject(aoi_run, zoom)
    return m
def ee_layer(m, image, vis, name, shown=True, opacity=1.0):
    m.addLayer(ee.Image(image), vis, name, shown, opacity)  # appears in the Layers panel (toggle + opacity)
    return m
print(f"{COUNTRY} · {SEASON} · {YEAR} · S1 {S1_ORBIT}")

## Planting-window estimation

In [ ]:
# --- planting dekad (onset) — cue-fusion green-up (main seasons) or rainfall onset (short rains) ---
from src import (utils, s2_preprocess as S2, s1_preprocess as S1, fusion_phenometrics as FZ,
                 ltn as LTN, planting_date as PD, wrsi_feedback as WR)
from run import crop_mask_image
kc, soil = utils.load_crop_coeffs()
rows={(r['country'],r['season']):r for r in utils.viable_products(utils.load_calendar('config/season_calendar.csv')) if r['crop'].lower()=='maize'}
r=rows[(COUNTRY,SEASON)]; ss,se=utils.sos_window_dekads(r['sos_detection_window']); mask=crop_mask_image(ee,COUNTRY,'maize',None)
if SEASON=='Short rains':
    pet=WR.pet_dekadal(ee,aoi_run,YEAR); ch=WR.chirps_dekadal(ee,aoi_run,YEAR)
    planting=WR.wrsi_onset(ee,ch,ss,se,pet_ic=pet).updateMask(mask).toInt16()
else:
    s2=S2.build_s2_dekadal(ee,aoi_run,YEAR); s1=S1.build_s1_dekadal(ee,aoi_run,YEAR,orbit=S1_ORBIT); fpar=FZ.add_fpar_dekadal(ee,aoi_run,YEAR)
    g=FZ.build_fused_greenness(ee,s2,s1,fpar); ltn=LTN.build_ltn_prior(ee,aoi_run,ss,se)
    sos=FZ.detect_sos(ee,g,mask,ss,se,ltn_sos=ltn,ltn_pad=2); planting=PD.sos_to_planting(ee,sos,'maize').toInt16()
print('planting dekad computed for', COUNTRY, SEASON)

In [ ]:
# 5+7 false-start gate (green-up seasons; short rains already carries the 25/20 mm rule)
if SEASON!='Short rains':
    ok=WR.dryspell_false_start(ee,aoi_run,planting,YEAR,dk_lo=ss,dk_hi=se+2); planting=planting.updateMask(ok)
print('valid maize pixels:', planting.reduceRegion(ee.Reducer.count(),aoi_run,250,maxPixels=int(1e13)).get('planting_dekad').getInfo())

In [ ]:
M=new_map()
ee_layer(M, planting.clip(aoi_run), {'min':ss,'max':se+3,'palette':['440154','3b528b','21908d','5dc863','fde725']}, f'Planting dekad — {SEASON}')
M   # geemap renders its own GEE-native Layers panel (toggle + opacity slider) — no extra layer control needed

## Planting-window statistics
Distribution of the estimated planting dekad over maize area, an agreement (**skill**) score against the FEWS/FAO calendar window, and a per-admin table + ranked bar.

In [ ]:
# ============================================================
#  Planting-window STATISTICS  (run after the map cell)
#  distribution graph · calendar-agreement skill · per-admin table & ranked bar
# ============================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
PIXEL_HA = 6.25                                   # area of one 250 m pixel
lab = utils.dekad_label                           # e.g. 9 -> "9\xb7Mar"

# ---- 1. AOI-wide planting-dekad distribution (area per dekad) --------------
hist = (planting.reduceRegion(ee.Reducer.frequencyHistogram(), aoi_run, 250,
        maxPixels=int(1e13)).get('planting_dekad').getInfo() or {})
H   = {int(round(float(k))): v for k, v in hist.items()}
dks = list(range(min(H) if H else ss, (max(H) if H else se+3) + 1))
cnt = np.array([H.get(d, 0) for d in dks], float)
area = cnt * PIXEL_HA
tot  = cnt.sum()
def wq(q):                                        # area-weighted quantile dekad
    c = np.cumsum(cnt); return int(np.array(dks)[np.searchsorted(c, tot*q)]) if tot else None
mode_dk = int(dks[int(np.argmax(cnt))]) if tot else None
mean_dk = float((cnt*np.array(dks)).sum()/tot) if tot else float('nan')

print(f"── {COUNTRY} \xb7 {SEASON} {YEAR} \xb7 planting-window statistics ──")
print(f"  maize area planted : {area.sum():,.0f} ha  ({int(tot):,} pixels)")
if tot:
    print(f"  modal dekad        : {lab(mode_dk)}")
    print(f"  median (p50) / mean: {lab(wq(0.5))}  /  {mean_dk:.1f}")
    print(f"  central 80% window : {lab(wq(0.1))}  →  {lab(wq(0.9))}   (spread {wq(0.9)-wq(0.1)} dekads)")
    inwin = area[[ss <= d <= se for d in dks]].sum()
    print(f"  SKILL — within FEWS/FAO calendar [{lab(ss)}–{lab(se)}]: {inwin/area.sum()*100:.0f}% of area")

# ---- 2. distribution bar chart (green = in calendar window, amber = outside)
fig, ax = plt.subplots(figsize=(8.4, 3.4))
ax.bar([lab(d) for d in dks], area/1000,
       color=['#3b7a57' if ss <= d <= se else '#c9a227' for d in dks])
ax.set_ylabel('maize area (000 ha)'); ax.set_xlabel('planting dekad')
ax.set_title(f'Planting-window distribution — {COUNTRY} {SEASON} {YEAR}')
ax.tick_params(axis='x', rotation=45)
for t in ax.get_xticklabels(): t.set_ha('right')
plt.tight_layout(); plt.show()

# ---- 3. per-admin planting window (GAUL level-1) --------------------------
admin = ZA.gaul_admin(ee, [GAUL_NAME[COUNTRY]], level=1).filterBounds(aoi_run)
feats = ZA.zonal_planting_stats(ee, planting, admin, scale=250).getInfo()['features']
def _key(props, suffix):                          # reduceRegions may or may not prefix with the band name 'pd'
    return next((k for k in props if k == suffix or k.endswith('_'+suffix)), None)
rows = []
if feats:
    pk = feats[0]['properties']
    kC, kMo, k10, k50, k90 = (_key(pk,'count'), _key(pk,'mode'), _key(pk,'p10'), _key(pk,'p50'), _key(pk,'p90'))
    for f in feats:
        p = f['properties']; n = (p.get(kC) or 0) if kC else 0
        if n < 20: continue                       # drop near-empty units
        gg = lambda k: lab(p[k]) if (k and p.get(k) is not None) else '—'
        rows.append(dict(Admin=p.get('ADM1_NAME','?'), _med=(p.get(k50) if k50 else None),
                         Modal=gg(kMo), Median=gg(k50), Early_p10=gg(k10), Late_p90=gg(k90),
                         Area_ha=round(n*PIXEL_HA)))
if not rows:
    print('  (no admin unit has >=20 maize pixels in aoi_run — set aoi_run = aoi in the config cell for the whole country)')
else:
    df = pd.DataFrame(rows).sort_values('_med', na_position='last').reset_index(drop=True)
    display(df.drop(columns='_med'))
    d2 = df.dropna(subset=['_med'])
    if len(d2):
        fig, ax = plt.subplots(figsize=(7.5, max(2.4, 0.34*len(d2))))
        ax.barh(d2['Admin'], d2['_med'], color='#3b528b'); ax.invert_yaxis()
        ax.set_xlabel('median planting dekad')
        for y, m in zip(range(len(d2)), d2['_med']):
            ax.text(m, y, ' '+lab(int(m)), va='center', fontsize=8)
        ax.set_title(f'Median planting dekad by admin — {COUNTRY} {SEASON}')
        plt.tight_layout(); plt.show()

## Farmer validation — 2024 MAM
Compares the estimated planting dekad with **farmer-reported** planting per Kenyan county (bias, MAE, ±2-dekad hit-rate). Runs only for **Kenya · Long rains**; uses your live estimate where `aoi_run` covers enough counties, otherwise the shipped validation CSV.

In [ ]:
# ============================================================
#  Farmer validation — 2024 MAM (Kenya long rains) planting dates
#  estimated vs observed farmer planting per county; bias / MAE / hit-rate
#  prefers a LIVE check of the estimate you just computed; falls back to the shipped CSV
# ============================================================
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
VAL_CSV = 'planting_validation_MAM_2024.csv'      # ships in the pipeline folder (on Drive)
if not (COUNTRY == 'Kenya' and 'ong' in SEASON):
    print('Farmer validation is for Kenya \xb7 Long rains (2024 MAM) only — skipping for', COUNTRY, SEASON)
elif not os.path.exists(VAL_CSV):
    print('Validation CSV not found at', os.path.abspath(VAL_CSV), '— upload it with the pipeline folder.')
else:
    obs = pd.read_csv(VAL_CSV)
    obs['County'] = obs['County'].astype(str).str.upper().str.strip()

    # LIVE: median estimated dekad per county from the planting image you just computed
    def _key(props, suffix): return next((k for k in props if k == suffix or k.endswith('_'+suffix)), None)
    adm = ZA.gaul_admin(ee, ['Kenya'], level=1).filterBounds(aoi_run)
    fe  = ZA.zonal_planting_stats(ee, planting, adm, scale=250).getInfo()['features']
    est = []
    if fe:
        kC, k50 = _key(fe[0]['properties'],'count'), _key(fe[0]['properties'],'p50')
        for f in fe:
            p = f['properties']; n = (p.get(kC) or 0) if kC else 0
            if n >= 20 and k50 and p.get(k50) is not None:
                est.append((str(p.get('ADM1_NAME','')).upper().strip(), float(p[k50])))
    est = pd.DataFrame(est, columns=['County','est_dk'])
    m = obs.merge(est, on='County', how='inner')
    if len(m) >= 5:
        src = f'LIVE — this run, {len(m)} counties in aoi_run'
    else:                                          # test box: too few counties -> use the shipped estimate
        m = obs.assign(est_dk=obs['modal_dekad']); src = f'shipped CSV estimate, {len(m)} counties'
    m = m.dropna(subset=['obs_dk','est_dk'])
    err = m['est_dk'] - m['obs_dk']                # estimated − observed (dekads)
    bias, mae, within2 = err.mean(), err.abs().mean(), (err.abs() <= 2).mean()*100
    hit = m['hit_rate'].mean()*100 if 'hit_rate' in m else float('nan')
    print(f"── Kenya \xb7 MAM 2024 farmer validation ({src}) ──")
    print(f"  bias (est−obs) : {bias:+.2f} dekads    MAE : {mae:.2f} dekads")
    print(f"  counties within \xb12 dekads : {within2:.0f}%    mean pixel hit-rate : {hit:.0f}%")

    # scatter: observed vs estimated, 1:1 line + ±2 dekad band
    lo = int(min(m['obs_dk'].min(), m['est_dk'].min())) - 1
    hi = int(max(m['obs_dk'].max(), m['est_dk'].max())) + 1
    fig, ax = plt.subplots(figsize=(4.7, 4.7))
    ax.fill_between([lo,hi], [lo-2,hi-2], [lo+2,hi+2], color='#3b7a57', alpha=0.12, label='\xb12 dekads')
    ax.plot([lo,hi], [lo,hi], 'k--', lw=1, label='1:1')
    ax.scatter(m['obs_dk'], m['est_dk'], s=30, color='#a50026', edgecolor='w', zorder=3)
    ax.set_xlim(lo,hi); ax.set_ylim(lo,hi); ax.set_aspect('equal')
    ax.set_xlabel('observed farmer planting (dekad)'); ax.set_ylabel('estimated planting (dekad)')
    ax.set_title('MAM 2024 · estimate vs farmers'); ax.legend(fontsize=8, loc='upper left')
    for _, r in m.iterrows():
        ax.annotate(r['County'].title(), (r['obs_dk'], r['est_dk']), fontsize=6, alpha=0.6,
                    xytext=(2,2), textcoords='offset points')
    plt.tight_layout(); plt.show()

    # worst-agreement counties
    worst = m.assign(err=err).reindex(err.abs().sort_values(ascending=False).index).head(8)
    cols = [c for c in ['County','obs_dk','est_dk','err','hit_rate'] if c in worst.columns]
    tbl = worst[cols].copy()
    tbl.columns = ['County','Observed dk','Estimated dk','Error (est−obs)'] + (['Hit-rate'] if 'hit_rate' in cols else [])
    display(tbl.reset_index(drop=True))

## County → ward drill-down
Ward-level (2024 MAM) validation, a nested **county→ward JSON**, and a per-county observed-vs-estimated bar. Change `COUNTY_PICK` to drill into any county.

In [ ]:
# ============================================================
#  County → Ward drill-down + ward-level validation   (2024 MAM, Kenya)
#  ward validation metrics · nested county→ward JSON · per-county grouped bar
# ============================================================
import os, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
WARD_CSV = 'planting_validation_MAM_ward_2024.csv'      # ships in the pipeline folder (on Drive)
if not os.path.exists(WARD_CSV):
    print('Ward CSV not found at', os.path.abspath(WARD_CSV), '— upload it with the pipeline folder.')
else:
    w = pd.read_csv(WARD_CSV).dropna(subset=['obs_dk','modal_dekad'])
    w['County'] = w['County'].astype(str).str.upper().str.strip()
    w['err'] = w['modal_dekad'] - w['obs_dk']              # estimated − observed (dekads)
    fw = w['Farmers (n)'].fillna(0).clip(lower=0)

    # ward-level validation, weighted by number of farmers surveyed
    bias = (w['err']*fw).sum()/max(fw.sum(),1); mae = (w['err'].abs()*fw).sum()/max(fw.sum(),1)
    within2 = (w['err'].abs() <= 2).mean()*100
    print(f"── Ward-level farmer validation · {len(w)} wards · {w['County'].nunique()} counties ──")
    print(f"  farmer-weighted bias {bias:+.2f} dk · MAE {mae:.2f} dk · wards within \xb12 dekads: {within2:.0f}%")

    # nested county -> ward JSON  {County: {Ward: {obs, est, err, farmers, n_px}}}
    nested = {}
    for _, r in w.iterrows():
        farmers = int(r['Farmers (n)']) if pd.notna(r['Farmers (n)']) else 0
        nested.setdefault(r['County'].title(), {})[str(r['Ward']).title()] = dict(
            obs=int(r['obs_dk']), est=int(r['modal_dekad']), err=int(r['err']),
            farmers=farmers, n_px=int(r['n_px']) if pd.notna(r.get('n_px')) else 0)
    with open('planting_ward_MAM_2024.json','w') as f: json.dump(nested, f, indent=1)
    print(f"  wrote planting_ward_MAM_2024.json ({len(nested)} counties, {len(w)} wards)")

    # ---- drill-down: pick a county -> observed vs estimated per ward --------
    COUNTY_PICK = 'BUNGOMA'                                # <-- change to any county
    sub = w[w['County'] == COUNTY_PICK.upper().strip()].sort_values('obs_dk')
    if not len(sub):
        print('No wards for', COUNTY_PICK, '· try one of:', ', '.join(sorted(w['County'].unique())))
    else:
        x = np.arange(len(sub)); bwid = 0.4
        fig, ax = plt.subplots(figsize=(max(6, 0.42*len(sub)), 4))
        ax.bar(x-bwid/2, sub['obs_dk'],      bwid, label='observed (farmers)', color='#3b7a57')
        ax.bar(x+bwid/2, sub['modal_dekad'], bwid, label='estimated',          color='#a50026')
        ax.set_xticks(x); ax.set_xticklabels(sub['Ward'].str.title(), rotation=60, ha='right', fontsize=7)
        ax.set_ylabel('planting dekad'); ax.legend(fontsize=8)
        ax.set_title(f'{COUNTY_PICK.title()} — planting dekad by ward (2024 MAM)')
        plt.tight_layout(); plt.show()
        show = sub[['Ward','obs_dk','modal_dekad','err','Farmers (n)']].rename(
            columns={'obs_dk':'Observed','modal_dekad':'Estimated','err':'Error (est−obs)','Farmers (n)':'Farmers'})
        display(show.reset_index(drop=True))

*Higher dekad = later planting. Export with `ee.batch.Export.image.toDrive(...)`; see `run.py` for batch runs.*